# Val Inspector v3
Run val samples or a real scan through the full pipeline (backbone → transformer → coarse → morph → fine registration).
For real scans there is no GT — pass dummy `gt_z=zeros` and `transform=eye`; only qualitative viz is meaningful.

In [3]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SNAPSHOT     = 'epoch-30.pth.tar'  # filename inside output/.../snapshots/
VAL_IDX      = 0                   # which val sample to inspect (0-indexed)
REAL_SCAN    = '2189.npy'          # .npy in this experiment dir (set None to skip)
DEVICE       = 'cuda'

# Real scan preprocessing — must match how the scan was acquired.
# 2189.npy: divide raw coords by 1/1.17 (≡ multiply by 1.17) to reach UHM meter-scale,
#            flip Z (scanner Z-axis points away from face), then center at origin.
REAL_SCAN_SCALE  = 1 / 1.17  # raw_pts /= REAL_SCAN_SCALE  (1.0 = no rescaling)
REAL_SCAN_FLIP_Z = True       # negate Z axis
REAL_SCAN_CENTER = True       # subtract centroid
# ─────────────────────────────────────────────────────────────────────────────

In [4]:
import os, sys

EXP_DIR  = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.dirname(os.path.dirname(EXP_DIR))
sys.path.insert(0, EXP_DIR)
sys.path.insert(0, ROOT_DIR)
os.chdir(EXP_DIR)

import torch
import numpy as np
import plotly.graph_objects as go
from functools import partial

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.modules.ops.transformation import apply_transform

from config_dowsampled import make_cfg
from dataset import train_valid_data_loader
from model import create_model
from loss import OverallLoss, Evaluator

print('EXP_DIR :', EXP_DIR)
print('ROOT_DIR:', ROOT_DIR)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
EXP_DIR : /workspace/src/experiments/geotransformer.faces.stage4.gse.k3.max.oacl.stage2.sinkhorn
ROOT_DIR: /workspace/src


In [5]:
cfg = make_cfg()

# calibrate neighbor_limits from train set, then get val loader
_, val_loader, neighbor_limits = train_valid_data_loader(
    cfg, distributed=False, val_aug_scale=1.0, val_aug_subsample=1.0
)
print('neighbor_limits :', neighbor_limits)
print('val samples     :', len(val_loader.dataset))

neighbor_limits : [60 25 29 31]
val samples     : 500


In [6]:
SNAP_DIR = os.path.join(
    ROOT_DIR, 'output',
    'geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn',
    'snapshots',
)
ckpt_path = os.path.join(SNAP_DIR, SNAPSHOT)

model = create_model(cfg).to(DEVICE)
model.neighbor_limits = neighbor_limits

ckpt = torch.load(ckpt_path, map_location=DEVICE)
sd   = ckpt.get('model', ckpt)
sd   = {k.replace('module.', ''): v for k, v in sd.items()}

model_keys = set(model.state_dict().keys())
missing  = model_keys - set(sd)
extra    = set(sd)    - model_keys
if missing: print(f'[warn] missing keys (random init): {len(missing)}')
if extra:   print(f'[info] extra keys ignored         : {len(extra)}')

model.load_state_dict(sd, strict=False)
model.eval()

loss_fn   = OverallLoss(cfg).to(DEVICE)
evaluator = Evaluator(cfg).to(DEVICE)

print(f'Loaded : {SNAPSHOT}  (epoch {ckpt.get("epoch", "?")})')
print(f'Keys matched: {len(set(sd) & model_keys)} / {len(model_keys)}')

Loaded : epoch-30.pth.tar  (epoch 30)
Keys matched: 316 / 316


---
## Val sample

In [7]:
collate_fn = partial(
    registration_collate_fn_stack_mode,
    num_stages=cfg.backbone.num_stages,
    voxel_size=cfg.backbone.init_voxel_size,
    search_radius=cfg.backbone.init_radius,
    neighbor_limits=neighbor_limits,
    precompute_data=False,
)

raw = val_loader.dataset[VAL_IDX]
data_dict = collate_fn([raw])
for k, v in data_dict.items():
    if isinstance(v, torch.Tensor):
        data_dict[k] = v.to(DEVICE)
    elif isinstance(v, list) and v and isinstance(v[0], torch.Tensor):
        data_dict[k] = [t.to(DEVICE) for t in v]

with torch.no_grad():
    output_dict = model(data_dict)

loss_dict = loss_fn(output_dict, data_dict, epoch=999, mode='val')
eval_dict = evaluator(output_dict, data_dict)

print('── Registration ──────────────────────')
print(f'  RRE   : {eval_dict["RRE"].item():.3f}°')
print(f'  RTE   : {eval_dict["RTE"].item():.4f} m')
print(f'  RMSE  : {eval_dict["RMSE"].item():.4f} m')
print(f'  RR    : {eval_dict["RR"].item():.3f}')
print(f'  PIR   : {eval_dict["PIR"].item():.3f}')
print(f'  IR    : {eval_dict["IR"].item():.3f}')
print('── Losses ────────────────────────────')
print(f'  morph : {loss_dict["m_loss"].item():.6f}')
print(f'  coarse: {loss_dict["c_loss"].item():.6f}')
print(f'  fine  : {loss_dict["f_loss"].item():.6f}')

── Registration ──────────────────────
  RRE   : 0.944°
  RTE   : 0.0039 m
  RMSE  : 0.0036 m
  RR    : 1.000
  PIR   : 0.965
  IR    : 0.913
── Losses ────────────────────────────
  morph : 0.005771
  coarse: 0.588902
  fine  : 2.281046


In [8]:
# ── Plotly helpers ─────────────────────────────────────────────────────────────
def pcd_trace(pts, color, name, size=2, opacity=0.7):
    if isinstance(pts, torch.Tensor):
        pts = pts.cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )

def show_pcd(traces, title=''):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=600,
        scene=dict(aspectmode='data'),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()

In [9]:
# ── Registration visualization ────────────────────────────────────────────────
ref_pts      = output_dict['ref_points']           # morphed ref
src_pts      = output_dict['src_points']           # raw src (model-level, stage-0)
est_tf       = output_dict['estimated_transform']
gt_tf        = data_dict['transform']
src_pred_aln = apply_transform(src_pts, est_tf)
src_gt_aln   = apply_transform(src_pts, gt_tf)

rre = eval_dict['RRE'].item()
rte = eval_dict['RTE'].item()

show_pcd(
    [pcd_trace(ref_pts, 'steelblue', 'ref (morphed)'),
     pcd_trace(src_pts, 'tomato',    'src (raw)')],
    f'Before alignment — val {VAL_IDX}',
)
show_pcd(
    [pcd_trace(ref_pts,      'steelblue', 'ref (morphed)'),
     pcd_trace(src_pred_aln, 'orange',    f'src pred  RRE={rre:.2f}° RTE={rte:.4f}m')],
    f'Predicted alignment — val {VAL_IDX}',
)
show_pcd(
    [pcd_trace(ref_pts,    'steelblue', 'ref (morphed)'),
     pcd_trace(src_gt_aln, 'seagreen',  'src GT transform')],
    f'GT alignment — val {VAL_IDX}',
)

In [10]:
# ── Morphing visualization ────────────────────────────────────────────────────
pred_morph = output_dict['morphed_full']        # morphed from pred_z
gt_morph   = output_dict['recon_gt_points']     # reconstructed from gt_z
m_loss     = loss_dict['m_loss'].item()

show_pcd(
    [pcd_trace(pred_morph, 'tomato',   'pred morph (pred z)'),
     pcd_trace(gt_morph,   'seagreen', 'GT morph   (gt z)')],
    f'Morphed ref overlay — morph_loss={m_loss:.6f}  val {VAL_IDX}',
)

---
## Real scan inference
No ground truth — qualitative only.  
Dummy `gt_z = zeros(32,100)` and `transform = eye(4)`.  
Model is in eval mode so `coarse_target` (which uses GT transform) is never called.

In [11]:
if REAL_SCAN is None:
    print('REAL_SCAN is None — skipping real scan section')
else:
    scan_path = os.path.join(EXP_DIR, REAL_SCAN)
    src_raw   = np.load(scan_path)
    src_real  = torch.from_numpy(src_raw[:, :3].astype(np.float32)).to(DEVICE)

    # Preprocessing: scale → optional Z-flip → optional center
    src_real /= REAL_SCAN_SCALE
    if REAL_SCAN_FLIP_Z:
        src_real[:, 2] *= -1
    if REAL_SCAN_CENTER:
        src_real -= src_real.mean(dim=0, keepdim=True)

    # mean reference face (generate_reference_geometry with z=0)
    with torch.no_grad():
        mean_ref = model.generate_reference_geometry(torch.zeros(32, 100, device=DEVICE))

    n_ref_r = mean_ref.shape[0]
    n_src_r = src_real.shape[0]

    data_dict_real = {
        'points'    : torch.cat([mean_ref, src_real], dim=0),
        'lengths'   : torch.tensor([n_ref_r, n_src_r], dtype=torch.int64, device=DEVICE),
        'features'  : torch.ones(n_ref_r + n_src_r, 1, device=DEVICE),
        'gt_z'      : torch.zeros(32, 100, device=DEVICE),    # dummy — no GT
        'transform' : torch.eye(4, device=DEVICE),            # dummy — no GT
    }

    with torch.no_grad():
        out_real = model(data_dict_real)

    print(f'Scan          : {REAL_SCAN}  →  {n_src_r} pts')
    print(f'Src range     : x=[{src_real[:,0].min():.3f}, {src_real[:,0].max():.3f}]'
          f'  y=[{src_real[:,1].min():.3f}, {src_real[:,1].max():.3f}]'
          f'  z=[{src_real[:,2].min():.3f}, {src_real[:,2].max():.3f}]')
    print(f'Mean ref      : {n_ref_r} pts')
    print(f'Morphed ref   : {out_real["morphed_full"].shape}')
    print(f'Est transform :\n{out_real["estimated_transform"].cpu().numpy()}')

Scan          : 2189.npy  →  5660 pts
Src range     : x=[-0.512, 0.558]  y=[-0.576, 0.594]  z=[-0.292, 0.283]
Mean ref      : 10788 pts
Morphed ref   : torch.Size([10788, 3])
Est transform :
[[-0.9905969   0.04824496 -0.12802503  0.01355845]
 [-0.03929575 -0.99666417 -0.07153116 -0.06528088]
 [-0.13104898 -0.06582762  0.989188    0.32798186]
 [ 0.          0.          0.          1.        ]]


In [12]:
if REAL_SCAN is not None:
    morphed_real  = out_real['morphed_full']
    est_tf_real   = out_real['estimated_transform']
    src_pts_real  = out_real['src_points']
    src_aln_real  = apply_transform(src_pts_real, est_tf_real)

    show_pcd(
        [pcd_trace(mean_ref,   'steelblue', 'mean ref'),
         pcd_trace(src_real,   'tomato',    'real scan (raw)')],
        f'{REAL_SCAN} — raw positions',
    )
    show_pcd(
        [pcd_trace(morphed_real, 'steelblue', 'morphed ref (pred z)'),
         pcd_trace(src_aln_real, 'orange',    'real scan — predicted alignment')],
        f'{REAL_SCAN} — predicted alignment (src aligned to morphed ref)',
    )
    show_pcd(
        [pcd_trace(mean_ref,    'steelblue', 'mean ref'),
         pcd_trace(morphed_real, 'tomato',   'morphed ref (pred z)')],
        f'{REAL_SCAN} — mean ref vs morphed ref (morph quality)',
    )